In [ ]:
# Change the branch to yours
# And choose an L4 GPU in Runtime > Change runtime type
BRANCH = "test-khalit"
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"


In [ ]:
# Generate a public key for SSH - add the printed result to your GitHub account
%%bash
set -e

mkdir -p /root/.ssh
chmod 700 /root/.ssh

if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi

ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts

echo "Public key:"
cat /root/.ssh/id_ed25519.pub


In [ ]:
# SSH test
%%bash
set -e
ssh -T git@github.com || true


In [ ]:
import os
os.environ["BRANCH"] = BRANCH
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR


In [ ]:
# Clone and checkout the branch indicated on the top
# After running, you should be able to see the repo in the Files in the left menu bar
%%bash
set -e

if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
git fetch origin
git checkout "$BRANCH"
git pull origin "$BRANCH"
git branch --show-current
git log --oneline -1


In [ ]:
# See GPU/CPU/RAM info
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"
echo -e "\n=== CPU Info ==="
lscpu | head -n 20
echo -e "\n=== Memory Info ==="
free -h


In [ ]:
# Install vLLM in a fresh runtime
%%bash
set -e
python -m pip uninstall -y sglang sgl-kernel flashinfer-python vllm torch torchvision torchaudio || true
python -m pip install -q -U uv pandas numpy requests tqdm
uv pip install -U vllm --torch-backend=auto


In [ ]:
# You should see a working vLLM installation
import torch
from importlib.metadata import version
print("torch:", torch.__version__)
print("vllm version:", version("vllm"))
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))


In [ ]:
# Stop the vLLM server when needed
!pkill -f "vllm.entrypoints" || true


In [ ]:
# Verify vLLM is gone
!ps aux | grep "vllm.entrypoints" | grep -v grep


In [ ]:
# Combined cleanup before launching vLLM
%%bash
pkill -f "vllm.entrypoints" || true
sleep 2
ps aux | grep "vllm.entrypoints" | grep -v grep || true


In [ ]:
# Launch the vLLM server on a different port
%%bash
cd /content/sglang
nohup python -m vllm.entrypoints.api_server   --tokenizer-mode auto   --model Qwen/Qwen2.5-1.5B-Instruct   --no-enable-log-requests   --port 21000   > /content/vllm_server.log 2>&1 &

echo $! > /content/vllm_server.pid
sleep 20
tail -n 50 /content/vllm_server.log


In [ ]:
# Check vLLM readiness
%%bash
curl -s http://127.0.0.1:21000/health || true
echo
tail -n 100 /content/vllm_server.log


In [ ]:
# Run multi-turn chat benchmark against vLLM once
# Send one dummy warmup request, then run the real benchmark
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
mkdir -p results
rm -f results/vllm_single.jsonl results/vllm_single_raw.json
curl -s http://127.0.0.1:21000/generate   -H "Content-Type: application/json"   -d '{"prompt": "1 2 3 4 5 6 7 8 9 10. This is a dummy warmup request for the vLLM server.", "temperature": 0.0, "max_tokens": 16, "n": 1}'
echo
python bench_multiturn_serving.py --backend vllm --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1 --result-file results/vllm_single.jsonl --raw-result-file results/vllm_single_raw.json --run-id single


In [ ]:
# Inspect the saved single-run vLLM result
%%bash
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
tail -n 1 results/vllm_single.jsonl


In [ ]:
# Run multi-turn chat benchmark against vLLM five times at parallel=1
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
mkdir -p results
rm -f results/vllm_p1_5runs.jsonl results/vllm_p1_run_*.json
for run_id in 1 2 3 4 5; do
  curl -s http://127.0.0.1:21000/generate     -H "Content-Type: application/json"     -d '{"prompt": "1 2 3 4 5 6 7 8 9 10. This is a dummy warmup request for the vLLM server.", "temperature": 0.0, "max_tokens": 16, "n": 1}' > /tmp/vllm_dummy_warmup.json
  python bench_multiturn_serving.py --backend vllm --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1 --result-file results/vllm_p1_5runs.jsonl --raw-result-file results/vllm_p1_run_${run_id}.json --run-id "$run_id"
done


In [ ]:
# Run multi-turn chat benchmark against vLLM five times at parallel=16
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
mkdir -p results
rm -f results/vllm_p16_5runs.jsonl results/vllm_p16_run_*.json
for run_id in 1 2 3 4 5; do
  curl -s http://127.0.0.1:21000/generate     -H "Content-Type: application/json"     -d '{"prompt": "1 2 3 4 5 6 7 8 9 10. This is a dummy warmup request for the vLLM server.", "temperature": 0.0, "max_tokens": 16, "n": 1}' > /tmp/vllm_dummy_warmup.json
  python bench_multiturn_serving.py --backend vllm --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 16 --result-file results/vllm_p16_5runs.jsonl --raw-result-file results/vllm_p16_run_${run_id}.json --run-id "$run_id"
done


In [ ]:
import json
from pathlib import Path

import pandas as pd

def summarize(prefix):
    base = Path('/content/sglang/690AB/eval/experiment_1/multi_turn_chat_short/results')
    rows = []
    for run_id in range(1, 6):
        with (base / f'{prefix}_run_{run_id}.json').open() as f:
            run = json.load(f)
        row = {
            'run_id': run['run_id'],
            'mean_e2e_latency_ms': run['metrics']['mean_e2e_latency_ms'],
            'p90_e2e_latency_ms': run['extended_metrics']['p90_e2e_latency_ms'],
            'p95_e2e_latency_ms': run['extended_metrics'].get('p95_e2e_latency_ms'),
            'request_throughput': run['metrics']['request_throughput'],
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    metrics = []
    metrics.append({'metric': 'mean_e2e_latency_ms', 'mean': df['mean_e2e_latency_ms'].mean(), 'std': df['mean_e2e_latency_ms'].std(ddof=1)})
    metrics.append({'metric': 'p90_e2e_latency_ms', 'mean': df['p90_e2e_latency_ms'].mean(), 'std': df['p90_e2e_latency_ms'].std(ddof=1)})
    metrics.append({'metric': 'p95_e2e_latency_ms', 'mean': df['p95_e2e_latency_ms'].mean(), 'std': df['p95_e2e_latency_ms'].std(ddof=1)})
    metrics.append({'metric': 'request_throughput', 'mean': df['request_throughput'].mean(), 'std': df['request_throughput'].std(ddof=1)})
    return df, pd.DataFrame(metrics)

print('vllm_p1')
df, summary = summarize('vllm_p1')
display(df)
display(summary)

print('vllm_p16')
df, summary = summarize('vllm_p16')
display(df)
display(summary)
